# 08. Listing Tools on the Azure AI Language MCP Server (MCP Client)

This notebook is an **MCP (Model Context Protocol) client** — it does not call Azure AI Language directly via
the `azure-ai-textanalytics` SDK (notebooks 05–07) or the REST API. Instead it connects, over the
**Streamable HTTP** MCP transport, to the **Azure AI Language service's own hosted MCP server endpoint**
(`https://<resource>.cognitiveservices.azure.com/language/mcp`) and asks it what tools it exposes.

**Important correction on scope:** unlike some other scripts in this course that connect to *custom* MCP
servers built on top of Azure Functions (as in earlier sections of this course), this script connects to
Azure AI Language's **own preview MCP endpoint** — a Microsoft-hosted surface that exposes Language service
capabilities (NER, PII, sentiment, etc.) as MCP tools an AI agent can call directly. It is not calling a
project-authored Azure Function; the "server" here is the Language resource itself.

**Difficulty:** Intermediate


## Prerequisites

**Python packages** (already covered by the repo's root `requirements.txt`):
- `mcp` — the Model Context Protocol Python SDK (`ClientSession`, `streamable_http_client`)
- `httpx` — async HTTP client used to attach auth headers to the MCP transport

No new installs needed for this notebook.

**Azure resources required:**
- An Azure AI Language resource with the MCP server preview feature enabled (`language/mcp` endpoint,
  `api-version=2025-11-15-preview` at time of writing — preview features/paths can change)

**Environment variables** (add to root `.env`):
```
AZURE_LANGUAGE_RESOURCE_NAME=<your-language-resource-name>
AZURE_LANGUAGE_KEY=<your-language-resource-key>
```
Note this uses just the bare resource *name* (not a full URL) because the script builds the MCP URL from it —
distinct from the full-endpoint-URL pattern used in notebooks 05–07.


## What You'll Learn

- What the Model Context Protocol (MCP) is used for: letting an AI agent discover and call tools over a
  standard protocol, instead of hardcoding SDK calls
- How to open an MCP session over the **Streamable HTTP** transport with custom auth headers
- How `ClientSession.initialize()` and `list_tools()` work in the MCP client lifecycle
- Why a service like Azure AI Language exposing itself as an MCP server matters for building tool-using agents
  (vs. calling the SDK/REST API directly, as in notebooks 05–07)


### Step 1 — Build the MCP server URL and auth headers

The Language service's MCP endpoint lives at a fixed path (`/language/mcp`) off your Cognitive Services
resource's `cognitiveservices.azure.com` hostname, with an explicit preview `api-version`. Authentication uses
the same `Ocp-Apim-Subscription-Key` header pattern used across all Azure Cognitive Services REST APIs (not
just Language) — it's passed via `httpx.AsyncClient(headers=...)` so every request the MCP transport makes
carries it automatically.

💡 **Exam tip:** `Ocp-Apim-Subscription-Key` is the standard subscription-key header name across Cognitive
Services REST APIs (Language, Speech, Vision, Translator, etc.) when you're not using an SDK's credential
object — recognize it as equivalent in purpose to `AzureKeyCredential` in the SDK notebooks (05–07), just at
the raw HTTP layer. MCP itself is a newer, cross-vendor protocol for exposing tools/capabilities to AI
agents/orchestrators — it isn't a core Azure AI Language feature the AI-102 blueprint historically tests in
detail, but understanding *what it's for* (standardized tool discovery/invocation for agents) is useful
context for the exam's Azure AI Agent Service / tool-calling topics.

🔄 **Alternatives:** Call the Language SDK/REST endpoints directly (notebooks 05–07) when you're writing
application code that knows exactly which operation it needs — reach for the MCP surface when you're building
a general-purpose agent that needs to *discover* available tools/capabilities dynamically at runtime.


In [ ]:
import os
import asyncio
import httpx
from dotenv import load_dotenv

from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

load_dotenv()

LANGUAGE_RESOURCE_NAME = os.environ["AZURE_LANGUAGE_RESOURCE_NAME"]
SUBSCRIPTION_KEY = os.environ["AZURE_LANGUAGE_KEY"]

LANGUAGE_MCP_URL = (
    f"https://{LANGUAGE_RESOURCE_NAME}.cognitiveservices.azure.com"
    f"/language/mcp?api-version=2025-11-15-preview"
)

### Step 2 — Open the MCP session and list available tools

This is standard MCP client boilerplate:
1. `httpx.AsyncClient(headers=..., timeout=...)` — the underlying HTTP transport, carrying the subscription
   key header on every request. Note the longer `read=300.0` timeout — MCP tool calls (not exercised in this
   notebook, just tool *listing*) can be long-running.
2. `streamable_http_client(url, http_client, terminate_on_close=True)` — opens the Streamable HTTP MCP
   transport, yielding read/write streams.
3. `ClientSession(read_stream, write_stream)` — wraps those streams in the higher-level MCP protocol client.
4. `await session.initialize()` — performs the MCP handshake (protocol version negotiation, capability
   exchange) — required before any other MCP call.
5. `await session.list_tools()` — asks the server to enumerate the tools it exposes; each `Tool` object has a
   `.name` and `.description`.

💡 **Exam tip:** This async context-manager nesting (`httpx.AsyncClient` → `streamable_http_client` →
`ClientSession`) is a standard MCP client shape you'll see across MCP examples in this repo's `10_mcp/`
chapter too — recognizing the pattern (transport → session → initialize → call) transfers across any MCP
server you connect to, Azure-hosted or otherwise.

🔄 **Alternatives:** MCP also supports a **stdio transport** (spawning a local server subprocess and
communicating over stdin/stdout) — used for locally-run MCP servers rather than a remote HTTPS endpoint like
this one. See `10_mcp/03_mcp_client/` in this repo for stdio-based client examples.


In [ ]:
async def list_language_mcp_tools():
    headers = {
        "Ocp-Apim-Subscription-Key": SUBSCRIPTION_KEY
    }

    timeout = httpx.Timeout(
        timeout=30.0,
        read=300.0
    )

    async with httpx.AsyncClient(headers=headers, timeout=timeout) as http_client:
        async with streamable_http_client(
            url=LANGUAGE_MCP_URL,
            http_client=http_client,
            terminate_on_close=True,
        ) as (read_stream, write_stream, _):

            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()

                result = await session.list_tools()

                print(f"Found {len(result.tools)} tool(s) on the Azure Language MCP server:\n")

                for tool in result.tools:
                    print(f"• {tool.name}")
                    print(f"  {tool.description}")
                    print()

### Step 3 — Run the async client

In a plain script this is triggered with `asyncio.run(list_language_mcp_tools())` under
`if __name__ == "__main__":`. In a Jupyter notebook there's already a running event loop, so `asyncio.run()`
would raise a `RuntimeError` — `await` the coroutine directly in a cell instead (Jupyter's IPython kernel
supports top-level `await`).

🔄 **Alternatives:** If you need this to run as a standalone script outside a notebook, keep the original
`asyncio.run(...)` + `if __name__ == "__main__":` guard shown in `08_mcp_server_list_tools.py`.


In [ ]:
# In a notebook kernel, use top-level await instead of asyncio.run()
await list_language_mcp_tools()

## Summary

This notebook connected to Azure AI Language's hosted MCP server over Streamable HTTP, authenticated with a
subscription key header, and listed the tools the server exposes — a discovery step an AI agent would perform
before deciding which tool(s) to invoke for a given task. Unlike notebooks 05–07, this notebook never calls a
Language *operation* directly; it only demonstrates the protocol-level handshake and tool enumeration.


## Try It Yourself

1. **Easy:** Print `tool.inputSchema` (if available on the `Tool` object) for each listed tool, to see what
   parameters each MCP tool expects.
2. **Intermediate:** Extend the script to actually **call** one of the listed tools via
   `session.call_tool(name, arguments)` — e.g. a PII redaction or NER tool — and print the result.
3. **Advanced:** Compare the tool list returned here against the operations you called directly via SDK in
   notebooks 05–07 (PII, language detection, NER) — write a short note on which SDK operations have an MCP
   tool equivalent, and what you'd gain/lose by routing through MCP instead of the SDK directly in an
   agent-based architecture.
